# Chapter 32 — Convolutional Networks and Computer Vision

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

digits = load_digits()
X_img = digits.images / 16.0            # (n, 8, 8), unflattened
X, y = digits.data / 16.0, digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                      stratify=y, random_state=0)
Xtr_img, Xte_img = Xtr.reshape(-1, 8, 8), Xte.reshape(-1, 8, 8)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# A convolution slides a small filter across an image, computing a dot
# product -- Chapter 9's dot product again -- at every position.
def convolve2d(img, kernel):
    kh, kw = kernel.shape
    h, w = img.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            patch = img[i:i+kh, j:j+kw]
            out[i, j] = np.sum(patch * kernel)          # the dot product
    return out

vertical_edge = np.array([[1, 0, -1],
                          [1, 0, -1],
                          [1, 0, -1]], dtype=float)

img = X_img[0]                          # the first training digit, 8x8
feat = convolve2d(img, vertical_edge)

print(f"input image      {img.shape}")
print(f"3x3 filter        {vertical_edge.shape}")
print(f"output feature map {feat.shape}   <- shrinks by kernel_size - 1")
print(f"\none output value, by hand, at position (2, 2):")
patch = img[2:5, 2:5]
print(f"  image patch:\n{patch.round(2)}")
print(f"  patch . filter (elementwise, summed) = {np.sum(patch * vertical_edge):.4f}")
print(f"  convolve2d agrees: {feat[2, 2]:.4f}")

### Block 2  (`c2.py`)

In [ ]:
# Why convolution instead of a fully connected layer: parameter count,
# and what each parameter is even allowed to depend on.
img_size = 8 * 8                       # this dataset
n_filters, k = 4, 3

fc_params = img_size * 32              # a modest 32-unit hidden layer
conv_params = n_filters * (k * k)      # four 3x3 filters, weights only

print(f"{'architecture':<28}{'parameters':>12}")
print(f"{'fully connected, 32 hidden':<28}{fc_params:>12,}")
print(f"{'conv, 4 filters of 3x3':<28}{conv_params:>12,}")
print(f"ratio: {fc_params / conv_params:.0f}x fewer parameters in the conv layer")

for side in (8, 32, 256):
    n = side * side
    print(f"\nat a {side}x{side} image:")
    print(f"  fully connected, 32 hidden: {n*32:>10,} parameters")
    print(f"  conv, 4 filters of 3x3:     {conv_params:>10,} parameters  (unchanged)")

### Block 3  (`c3.py`)

In [ ]:
# Pooling shrinks the feature map and buys a little tolerance to small
# shifts: the same loudest signal survives even if it moves by a pixel.
def max_pool2d(feat, size=2):
    h, w = feat.shape
    oh, ow = h // size, w // size
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = feat[i*size:(i+1)*size, j*size:(j+1)*size].max()
    return out

feat = convolve2d(X_img[0], vertical_edge)
pooled = max_pool2d(feat, size=2)
print(f"feature map     {feat.shape}")
print(f"after 2x2 pool  {pooled.shape}")

# shift the image by one pixel and compare the pooled output
shifted = np.roll(X_img[0], 1, axis=1)
feat_shifted = convolve2d(shifted, vertical_edge)
pooled_shifted = max_pool2d(feat_shifted, size=2)

diff_raw = np.abs(feat[:5, :5] - feat_shifted[:5, :5]).mean()
diff_pooled = np.abs(pooled - pooled_shifted).mean()
print(f"\nafter shifting the image by one pixel:")
print(f"  mean change in the raw feature map: {diff_raw:.4f}")
print(f"  mean change after pooling:          {diff_pooled:.4f}")

### Block 4  (`c4.py`)

In [ ]:
# The backward pass through a convolution, derived and then checked
# exactly as Chapter 10 prescribed. This is the last hand-derived
# gradient in this book, and it is noticeably more work than any before
# it -- which is itself the argument for what comes after this chapter.
def conv_backward(dout, img, kernel):
    kh, kw = kernel.shape
    dkernel = np.zeros_like(kernel)
    for di in range(kh):
        for dj in range(kw):
            dkernel[di, dj] = np.sum(
                dout * img[di:di+dout.shape[0], dj:dj+dout.shape[1]])

    flipped = kernel[::-1, ::-1]
    pad_h, pad_w = kh - 1, kw - 1
    dout_padded = np.pad(dout, ((pad_h, pad_h), (pad_w, pad_w)))
    dimg = convolve2d(dout_padded, flipped)
    return dimg, dkernel

img = X_img[3]
kernel = vertical_edge.copy()
feat = convolve2d(img, kernel)
dout = np.random.default_rng(32).normal(size=feat.shape)   # a stand-in upstream gradient

dimg, dkernel = conv_backward(dout, img, kernel)
print(f"dkernel shape {dkernel.shape}   dimg shape {dimg.shape}")

# numerical check, Chapter 10's method, on three kernel entries and
# three image pixels
eps = 1e-5
def loss(img_, kernel_):
    return np.sum(convolve2d(img_, kernel_) * dout)

print(f"\n{'target':>14}{'analytic':>12}{'numerical':>12}{'match':>8}")
for (di, dj) in [(0, 0), (1, 1), (2, 2)]:
    orig = kernel[di, dj]
    kernel[di, dj] = orig + eps; lp = loss(img, kernel)
    kernel[di, dj] = orig - eps; lm = loss(img, kernel)
    kernel[di, dj] = orig
    numeric = (lp - lm) / (2 * eps)
    match = abs(numeric - dkernel[di, dj]) < 1e-4
    print(f"kernel{(di,dj)}{dkernel[di,dj]:>12.6f}{numeric:>12.6f}{str(match):>8}")

for (pi, pj) in [(1, 1), (4, 4), (6, 6)]:
    orig = img[pi, pj]
    img[pi, pj] = orig + eps; lp = loss(img, kernel)
    img[pi, pj] = orig - eps; lm = loss(img, kernel)
    img[pi, pj] = orig
    numeric = (lp - lm) / (2 * eps)
    match = abs(numeric - dimg[pi, pj]) < 1e-4
    print(f"pixel{(pi,pj)} {dimg[pi,pj]:>12.6f}{numeric:>12.6f}{str(match):>8}")

### Block 5  (`c5.py`)

In [ ]:
# A minimal CNN trained on the real data: four learned filters, max
# pooling, a fully connected output layer. The forward and backward
# passes are the vectorized, verified equivalent of Step 1's loop and
# Step 4's derivation, applied to a whole batch at once.
from numpy.lib.stride_tricks import sliding_window_view

def conv_forward(imgs, filters):
    windows = sliding_window_view(imgs, (3, 3), axis=(1, 2))
    return np.einsum('nijhw,fhw->nfij', windows, filters), windows

def conv_backward(dout, windows, filters):
    dfilters = np.einsum('nfij,nijhw->fhw', dout, windows)
    flipped = filters[:, ::-1, ::-1]
    pad = filters.shape[1] - 1
    dout_p = np.pad(dout, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    dwindows = sliding_window_view(dout_p, (3, 3), axis=(2, 3))
    dimgs = np.einsum('nfijhw,fhw->nij', dwindows, flipped)
    return dimgs, dfilters

def pool_forward(feats, size=2):
    n, nf, h, w = feats.shape
    r = feats.reshape(n, nf, h // size, size, w // size, size)
    out = r.max(axis=(3, 5))
    mask = (r == out[:, :, :, None, :, None])
    return out, mask

def pool_backward(dout, mask, size=2):
    n, nf, oh, ow = dout.shape
    d = dout[:, :, :, None, :, None] * mask
    return d.reshape(n, nf, oh * size, ow * size)

r = np.random.default_rng(32)
n_filters = 4
filters = r.normal(0, np.sqrt(2 / 9), (n_filters, 3, 3))
D_flat = n_filters * 3 * 3
Wf = r.normal(0, np.sqrt(2 / D_flat), (D_flat, 10))
bf = np.zeros(10)
Ytr = np.eye(10)[ytr]
eta = 0.3

print(f"{'epoch':>7}{'train loss':>13}{'test accuracy':>15}")
for epoch in range(151):
    conv_out, windows = conv_forward(Xtr_img, filters)
    relu_out = np.maximum(0, conv_out)
    pool_out, mask = pool_forward(relu_out)
    flat = pool_out.reshape(len(Xtr_img), -1)
    p = softmax(flat @ Wf + bf)
    loss = -np.sum(Ytr * np.log(p + 1e-12)) / len(Xtr_img)

    dscore = (p - Ytr) / len(Xtr_img)
    dWf = flat.T @ dscore
    dbf = dscore.sum(0)
    dflat = dscore @ Wf.T
    dpool = dflat.reshape(pool_out.shape)
    drelu = pool_backward(dpool, mask)
    dconv = drelu * (conv_out > 0)
    _, dfilters = conv_backward(dconv, windows, filters)

    Wf -= eta * dWf; bf -= eta * dbf
    filters -= eta * dfilters

    if epoch % 30 == 0:
        c_te, _ = conv_forward(Xte_img, filters)
        p_te, m_te = pool_forward(np.maximum(0, c_te))
        flat_te = p_te.reshape(len(Xte_img), -1)
        pred = softmax(flat_te @ Wf + bf).argmax(1)
        acc = (pred == yte).mean()
        print(f"{epoch:>7}{loss:>13.4f}{acc:>15.4f}")

c_te, _ = conv_forward(Xte_img, filters)
p_te, _ = pool_forward(np.maximum(0, c_te))
final_pred = softmax(p_te.reshape(len(Xte_img), -1) @ Wf + bf).argmax(1)
cnn_acc = (final_pred == yte).mean()
print(f"\nfinal CNN test accuracy: {cnn_acc:.4f}")

### Block 6  (`c6.py`)

In [ ]:
# Raw accuracy is not the only axis that matters. Shift every test image
# by one pixel and see which architecture degrades less: this is what
# the parameter-sharing argument actually predicts.
def cnn_predict(imgs):
    c, _ = conv_forward(imgs, filters)
    p, _ = pool_forward(np.maximum(0, c))
    return softmax(p.reshape(len(imgs), -1) @ Wf + bf).argmax(1)

Xte_shifted = np.roll(Xte_img, 1, axis=2)             # every test image, shifted right

cnn_acc_orig = (cnn_predict(Xte_img) == yte).mean()
cnn_acc_shift = (cnn_predict(Xte_shifted) == yte).mean()

fc_baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
fc_baseline.fit(Xtr, ytr)
fc_acc_orig = fc_baseline.score(Xte, yte)
fc_acc_shift = fc_baseline.score(Xte_shifted.reshape(len(Xte), -1), yte)

print(f"{'model':<28}{'original':>10}{'shifted':>10}{'drop':>9}")
print(f"{'fully connected (logreg)':<28}{fc_acc_orig:>10.4f}{fc_acc_shift:>10.4f}"
      f"{fc_acc_orig-fc_acc_shift:>9.4f}")
print(f"{'CNN, 4 filters':<28}{cnn_acc_orig:>10.4f}{cnn_acc_shift:>10.4f}"
      f"{cnn_acc_orig-cnn_acc_shift:>9.4f}")